# 05 - Model experiments

Runs the controlled experiment ladder (EXP-00 ... EXP-07) with **identical stratified 5-fold splits** for every candidate. All preprocessing that learns from data (median imputation, PCA) is refitted inside each training fold.

Nothing is assumed to help: each addition (audio, text, embeddings, ensembling) must beat the simpler alternative in cross-validation.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
from src.config import load_config, repo_dir
from src.pipeline import get_context

# Dataset location: leave None to use the SHL_DATA_ROOT environment variable (or config.data_root),
# or set it here, e.g. DATA_ROOT = r"C:\Users\USERNAME\path\Dataset_Final"
DATA_ROOT = None
try:
    ctx = get_context(DATA_ROOT); cfg = ctx.cfg
    print("Data root:", ctx.paths.root)
except Exception as exc:
    ctx, cfg = None, load_config()
    print("Data root NOT configured ->", exc)
PENDING = "Results will be populated after local execution."
TABLES = repo_dir(cfg, "reports_dir") / "tables"
LOG_PATH = repo_dir(cfg, "experiments_dir") / "experiment_log.csv"
def have(*paths): return all(Path(p).exists() for p in paths)

In [ ]:
import yaml
from src.models.train import load_experiments, run_experiments, read_log, enabled_groups
exps = load_experiments(ROOT / cfg["experiments_file"])
print(pd.DataFrame([{"id": e["id"], "feature_set": e.get("feature_set", "-"), "models": e.get("models", e.get("type"))} for e in exps]))

In [ ]:
RUN_CV = False   # True runs all experiments here (minutes). Equivalent CLI: python scripts/train.py
if RUN_CV and ctx is not None:
    from src.pipeline import load_train_matrix
    X, y = load_train_matrix(ctx)
    log = run_experiments(X, y, cfg, exps, LOG_PATH, TABLES / "oof_predictions.csv")

## Results

In [ ]:
if have(LOG_PATH):
    log = read_log(LOG_PATH)
    show = log.sort_values("cv_rmse_mean")[["experiment_id", "feature_set", "model", "cv_rmse_mean", "cv_rmse_std", "cv_pearson_mean", "cv_pearson_std", "training_rmse"]]
    display(show.round(4))
    from src.reporting import plot_model_comparison
    plot_model_comparison(log); plt.show()
else:
    print(PENDING)

## Controlled comparisons
* **A** handcrafted text (EXP-03) vs **B** embeddings (EXP-05) vs **C** text + embeddings (EXP-05b)
* Does fusing audio + text (EXP-04) beat each alone (EXP-02, EXP-03)?
* Does adding embeddings to the fusion (EXP-06) help?
* Does the ensemble (EXP-07) beat the best single model?

Differences smaller than the fold-to-fold standard deviation are *not* evidence of improvement.

In [ ]:
if have(LOG_PATH):
    best_per = log.sort_values("cv_rmse_mean").groupby("experiment_id").first()[["feature_set", "model", "cv_rmse_mean", "cv_rmse_std", "cv_pearson_mean"]]
    display(best_per.round(4))